In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

In [ ]:
""" def check_pin_visibility(image_path):
    img = cv2.imread(image_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Gaussian Blur (Common step)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)

    # Adaptive Threshold
    # Block Size 25, Constant 10 (Fine tune these!)
    adaptive_result = cv2.adaptiveThreshold(blurred, 255, 
                                            cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
                                            cv2.THRESH_BINARY_INV, 45, 10)

    if os.path.exists("output"):
        cv2.imwrite(f"output/adaptive_{image_path.split('/')[-1]}", adaptive_result)
    else:
        os.mkdir("output")
        cv2.imwrite(f"output/adaptive_{image_path.split('/')[-1]}", adaptive_result)

for img in os.listdir("./data/socket/"):
    check_pin_visibility(f"./data/socket/{img}")  """

In [ ]:
def extract_precise_grid_hough(binary_path):
    binary = cv2.imread(binary_path, cv2.IMREAD_GRAYSCALE)
    
    circles = cv2.HoughCircles(
        binary, 
        cv2.HOUGH_GRADIENT, 
        dp=1,           # Inverse ratio of accumulator resolution
        minDist=10,     # Minimum distance between circle centers
        param1=60,      # Upper threshold for edge detection
        param2=10,      # Accumulator threshold for center detection
        minRadius=5,    # Minimum circle radius
        maxRadius=20    # Maximum circle radius
    )
    
    coords = []
    if circles is not None:
        circles = np.round(circles[0, :]).astype("int")
        coords = [(x, y) for x, y in circles[:, :2]]
        print(f"Found {len(coords)} circles with Hough")
    else:
        print("No circles found with Hough!")
    
    return coords

def extract_precise_grid(binary_path):
    binary = cv2.imread(binary_path, cv2.IMREAD_GRAYSCALE)
    
    template_size = 30  # Bigger template
    template = np.zeros((template_size, template_size), dtype=np.uint8)
    cv2.circle(template, (template_size//2, template_size//2), 8, 255, -1)  # Solid circle
    cv2.circle(template, (template_size//2, template_size//2), 12, 0, 3)    # Black ring around it
    
    result = cv2.matchTemplate(binary, template, cv2.TM_CCOEFF_NORMED)
    
    # MUCH higher threshold
    threshold = 0.8  # Increased from 0.7
    locations = np.where(result >= threshold)
    
    # Remove duplicate detections (Non-Maximum Suppression)
    coords_raw = list(zip(locations[1], locations[0]))
    
    # Filter out coordinates that are too close to each other
    coords = []
    min_distance = 20  # Minimum pixels between pin centers
    
    for x, y in coords_raw:
        # Check if this coord is far enough from existing ones
        too_close = False
        for existing_x, existing_y in coords:
            distance = np.sqrt((x - existing_x)**2 + (y - existing_y)**2)
            if distance < min_distance:
                too_close = True
                break
        
        if not too_close:
            coords.append((x, y))
    
    print(f"Found {len(coords)} pins after filtering")
    return coords

def extract_aligned_pins(image_path, coords, crop_size=32):
    # Extract pins with better preprocessing
    img = cv2.imread(image_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # CLAHE for better contrast
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    enhanced = clahe.apply(gray)
    
    pins = []
    for i, (cx, cy) in enumerate(coords):
        half = crop_size // 2
        
        # Better boundary checking
        if (cy-half >= 0 and cx-half >= 0 and cy+half < enhanced.shape[0] and cx+half < enhanced.shape[1]):
            crop = enhanced[cy-half:cy+half, cx-half:cx+half]
            
            # Normalize each pin individually
            crop = cv2.normalize(crop, None, 0, 255, cv2.NORM_MINMAX)
            crop = cv2.GaussianBlur(crop, (3,3), 0)
            
            pins.append((crop, i, cx, cy))

    return pins

""" def extract_aligned_pins(image, center_x, center_y, size=64):
    # Extracts a fixed-size square crop around the pin center.
    h, w = image.shape[:2]
    x1 = int(center_x - size // 2)
    y1 = int(center_y - size // 2)
    x2 = x1 + size
    y2 = y1 + size

    # Check bounds
    if x1 < 0 or y1 < 0 or x2 > w or y2 > h:
        return None  # Skip pins on the very edge that might be cut off

    crop = image[y1:y2, x1:x2]
    return crop """

In [ ]:
def filter_good_pins(pins, variance_threshold=100):
    """Remove blurry/bad quality pins and not pins (outside pins area in socket)"""
    good_pins = []
    
    for crop, idx, x, y in pins:
        # Check if pin is in focus (Laplacian variance)
        variance = cv2.Laplacian(crop, cv2.CV_64F).var()
        
        if variance > variance_threshold:
            good_pins.append((crop, idx, x, y))
    
    max_dist_pins = 20 # max distance from one pins center to another
    final_good_pins = []
    for i in range(len(good_pins)):
        crop_i, idx_i, x_i, y_i = good_pins[i]
        too_close = False
        for j in range(len(final_good_pins)):
            crop_j, idx_j, x_j, y_j = final_good_pins[j]
            dist = np.sqrt((x_i - x_j)**2 + (y_i - y_j)**2)
            if dist < max_dist_pins:
                too_close = True
                break
        if not too_close:
            final_good_pins.append(good_pins[i])
    good_pins = final_good_pins
    print(f"Filtered {len(pins)} -> {len(good_pins)} quality pins")
    return good_pins

def visualize_pin_quality(pins):
    fig, axes = plt.subplots(4, 8, figsize=(16, 8))
    for i, (crop, idx, x, y) in enumerate(pins[:32]):
        row, col = i // 8, i % 8
        axes[row, col].imshow(crop, cmap='gray')
        axes[row, col].set_title(f"{idx}")
        axes[row, col].axis('off')
    plt.show()

def visualize_detection_on_original(image_path, pins):
    img = cv2.imread(image_path)
    vis_img = img.copy()
    crop_size = 32
    coords = [(x, y) for (_, _, x, y) in pins]

    # Draw Bounding Boxes
    half_size = crop_size // 2
    h, w = vis_img.shape[:2] # Get image dimensions

    for (x, y) in coords:
        # Calculate bounding box coordinates
        x1 = x - half_size
        y1 = y - half_size
        x2 = x + half_size
        y2 = y + half_size

        # Green (0, 255, 0), Thickness 2 px
        cv2.rectangle(vis_img, (x1, y1), (x2, y2), (0, 255, 0), 2)
        
        # Optional: Draw a small red dot at the exact center
        cv2.circle(vis_img, (x, y), 2, (0, 0, 255), -1)
        
    # Convert BGR (OpenCV) to RGB (Matplotlib)
    plt.figure(figsize=(12, 8))
    plt.imshow(cv2.cvtColor(vis_img, cv2.COLOR_BGR2RGB))
    plt.title(f"Detection Evaluation: {len(coords)} Pins Detected")
    plt.axis('off')
    plt.show()
    
    return vis_img

In [ ]:
# Use the coordinates we found in the previous step
# coords = [(x1,y1), (x2,y2), ...] 

def create_anomalib_dataset(image_path, coords, output_dir="datasets/socket_pins"):
    img = cv2.imread(image_path)
    
    # Create folders
    train_dir = os.path.join(output_dir, "train/good")
    test_dir = os.path.join(output_dir, "test/good")
    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(test_dir, exist_ok=True)
    
    # Crop settings
    crop_size = 32  # 32x32 is standard for small features like pins
    half = crop_size // 2
    
    for i, (cx, cy) in enumerate(coords):
        # Boundary check
        if cy-half < 0 or cx-half < 0: continue
        
        crop = img[cy-half:cy+half, cx-half:cx+half]
        
        # Save 80% to Train, 20% to Test
        if i % 5 == 0:
            cv2.imwrite(f"{test_dir}/pin_{i}.png", crop)
        else:
            cv2.imwrite(f"{train_dir}/pin_{i}.png", crop)
            
    print(f"Dataset generated at {output_dir}")


for img in os.listdir("./data/socket/"):
    print(f"Processing {img}...")
    
    # Get coordinates from binary image
    binary_path = f"output/adaptive_{img}"
    coords = extract_precise_grid_hough(binary_path)
    
    # Extract pins from original image
    original_path = f"./data/socket/{img}"
    pins = extract_aligned_pins(original_path, coords)

    # Might crop 64x64 from the high-res image, then resize up, or crop larger.
    """ crop = extract_aligned_pins(original_path, center_x, center_y, size=224)
    if crop is not None:
        # Resize to standard size for ML model (usually 224 or 256)
        crop_resized = cv2.resize(crop, (224, 224))
        cv2.imwrite(f"output/aligned_{img}", crop_resized) """
    
    # Filter and visualize
    pins = filter_good_pins(pins) # Need to work on this
    visualize_pin_quality(pins)
    visualize_detection_on_original(original_path, pins)
    
    # Create dataset
    create_anomalib_dataset(original_path, coords)